In [3]:
import io
import numpy as np
import pandas as pd
from google.colab import files

# =====================================================================
# 1. COLUMN CONFIGURATION
# =====================================================================
# Mapping your exact 7-column layout keys
ID_COLUMN = "CustomerID"
TEXT_COLUMN = "Gender"
CATEGORY_COLUMN = "Contract Length"  # Changed from DATE to CATEGORY to match data type
NUMERIC_COLUMN = "Total Spend"

# =====================================================================
# 2. UPLOAD AND INGEST EXTERNAL CSV
# =====================================================================
print("--- [STEP 1/3] IMPORT RAW DATASET ---")
print("Please select and upload your downloaded messy CSV file:")

try:
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
    df_raw = pd.read_csv(io.BytesIO(uploaded[file_name]))

    print(f"\n✓ Successfully loaded: {file_name}")
    print(
        f"Initial Dataset Dimensions: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.\n"
    )
    print("--- First 3 Rows of Raw Data ---")
    print(df_raw.head(3))
except Exception as e:
    print(f"\n❌ Error uploading file: {e}")
    print("Please ensure you are uploading a valid .csv file format.")

# =====================================================================
# 3. DATA CLEANING & STRUCTURAL VALIDATION PIPELINE
# =====================================================================
def clean_and_validate_data(df):
    # Establish a deep copy to preserve original raw state
    df_clean = df.copy()

    print("\n--- [STEP 2/3] RUNNING VALIDATION PIPELINE ---\n")

    # [TASK 1] Structural Validation: Header Standardization
    # Strip whitespace, lower characters, replace gaps/dots/hyphens with underscores
    df_clean.columns = (
        df_clean.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace(".", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    print("✓ Structural Validation: Column headers standardized to snake_case.")

    # Dynamically match our target columns to their new standardized names
    id_col = ID_COLUMN.strip().lower().replace(" ", "_")
    text_col = TEXT_COLUMN.strip().lower().replace(" ", "_")
    cat_col = CATEGORY_COLUMN.strip().lower().replace(" ", "_")
    num_col = NUMERIC_COLUMN.strip().lower().replace(" ", "_")

    # [TASK 2] Remove Duplicate Entries
    initial_rows = len(df_clean)
    df_clean.drop_duplicates(inplace=True)
    dropped_duplicates = initial_rows - len(df_clean)
    print(f"✓ Deduplication: Removed {dropped_duplicates} identical row record(s).")

    # [TASK 3] Clean and Cast Inconsistent Data Types (Numeric)
    if num_col in df_clean.columns:
        # Strip currency symbols, commas, or unexpected text strings
        df_clean[num_col] = (
            df_clean[num_col]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        # Convert to float numbers; set completely unparseable text items to NaN
        df_clean[num_col] = pd.to_numeric(df_clean[num_col], errors="coerce")
        print(f"✓ Data Types: Cleansed and converted '{num_col}' into numeric floats.")

    # [TASK 4] Handle Missing Values (Imputation & Dropping)
    # Strategy A: Strategic deletion if critical primary key identifiers are missing
    if id_col in df_clean.columns:
        df_clean.dropna(subset=[id_col], inplace=True)

    # Strategy B: Impute missing entries in the numeric column using column Median
    if num_col in df_clean.columns:
        median_value = df_clean[num_col].median()
        # In case the entire column is empty/NaN, default fill to 0
        if pd.isna(median_value):
            median_value = 0.0
        df_clean[num_col] = df_clean[num_col].fillna(median_value)
        print(f"✓ Missing Values: Imputed empty numerical values using median ({median_value}).")

    # [TASK 5] Inconsistent Formats (Text Standardization)
    # Standardize string capitalization structures for categorical features
    if text_col in df_clean.columns:
        df_clean[text_col] = df_clean[text_col].astype(str).str.strip().str.title()

    if cat_col in df_clean.columns:
        df_clean[cat_col] = df_clean[cat_col].astype(str).str.strip().str.title()
        print(f"✓ Inconsistent Formats: Stripped whitespace and standardized text patterns for '{text_col}' and '{cat_col}'.")

    # Re-index clean rows
    df_clean.reset_index(drop=True, inplace=True)

    return df_clean

# Execute the automation pipeline
try:
    df_final = clean_and_validate_data(df_raw)

    print("\n--- [STEP 3/3] EXPORT VALIDATED DATASETS ---")
    print("--- First 5 Rows of Cleaned Dataset ---")
    print(df_final.head(5))

    # Save outputs back out into local workspace files
    df_raw.to_csv("raw_dataset.csv", index=False)
    df_final.to_csv("cleaned_dataset.csv", index=False)
    print(
        "\n✓ Final verification files written out cleanly as 'raw_dataset.csv' and 'cleaned_dataset.csv'!"
    )
except NameError:
    print("❌ Pipeline execution halted. Please upload a dataset file successfully first.")


--- [STEP 1/3] IMPORT RAW DATASET ---
Please select and upload your downloaded messy CSV file:


Saving customer_churn_dataset.csv to customer_churn_dataset.csv

✓ Successfully loaded: customer_churn_dataset.csv
Initial Dataset Dimensions: 20000 rows, 11 columns.

--- First 3 Rows of Raw Data ---
   customer_id  tenure  monthly_charges  total_charges        contract  \
0            1      52            54.20        2818.40  Month-to-month   
1            2      15            35.28         529.20  Month-to-month   
2            3      72            78.24        5633.28  Month-to-month   

  payment_method internet_service tech_support online_security  support_calls  \
0         Credit              DSL           No             Yes              1   
1          Debit              DSL           No              No              2   
2          Debit              DSL           No              No              0   

  churn  
0    No  
1    No  
2    No  

--- [STEP 2/3] RUNNING VALIDATION PIPELINE ---

✓ Structural Validation: Column headers standardized to snake_case.
✓ Deduplication: Rem

# Internship Task Assignment: Data Cleaning & Structural Validation Pipeline

**Intern Profile:** Faizan Fayaz  
**Objective:** Parse, transform, and systematically structural-validate messy unstructured source variables into a clean, normalized schema ready for business intelligence tools.

---

### Production Pipeline Remediation Breakdown:

1. **Structural Schema Validation:** Standardized column headers by removing trailing spaces and converting miscellaneous spacing markers into normalized `snake_case` notation.
2. **Data Deduplication:** Tracked down and dropped redundant rows to prevent duplicated calculations during statistical metrics analysis.
3. **Data Type Casting:** Cleaned formatting artifacts (currency symbols, special characters) from numeric fields to safely cast string values into operational floats.
4. **Missing Value Management:** Handled missing rows by dropping lines lacking core unique IDs, and used statistical column medians to fill numeric gaps without introducing skew.
5. **String Standardizations:** Used string formatting across text entries to fix irregular text casing and stripped hidden trailing whitespaces.